# MABe Training Notebook 

In [ ]:
# 导入依赖库：数据处理、模型训练、并行计算与评价指标

import datetime
import gc
import itertools
import json
import re
import sys
import time
import traceback
from collections import defaultdict
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm
from metric import score

In [ ]:
# 全局配置：路径、关键字段、身体关键点、行为类别与模型参数

DATA_ROOT = Path('./data')
TRAIN_POSE_DIR = DATA_ROOT / 'train_tracking'
TRAIN_LABEL_DIR = DATA_ROOT / 'train_annotation'
TEST_POSE_DIR = DATA_ROOT / 'test_tracking'
ARTIFACT_DIR = Path('./working')
KEY_COLUMNS = ['video_id', 'agent_mouse_id', 'target_mouse_id', 'video_frame']
KEYPOINT_NAMES = ['ear_left', 'ear_right', 'nose', 'neck', 'body_center', 'lateral_left', 'lateral_right', 'hip_left', 'hip_right', 'tail_base', 'tail_tip']
SELF_ACTIONS = ['biteobject', 'climb', 'dig', 'exploreobject', 'freeze', 'genitalgroom', 'huddle', 'rear', 'rest', 'run', 'selfgroom']
PAIR_ACTIONS = ['allogroom', 'approach', 'attack', 'attemptmount', 'avoid', 'chase', 'chaseattack', 'defend', 'disengage', 'dominance', 'dominancegroom', 'dominancemount', 'ejaculate', 'escape', 'flinch', 'follow', 'intromit', 'mount', 'reciprocalsniff', 'shepherd', 'sniff', 'sniffbody', 'sniffface', 'sniffgenital', 'submit', 'tussle']
scales = np.logspace(np.log10(50), np.log10(5000), num=7, dtype=int).tolist()
MODEL_CONFIG = {'speed_time_scales': scales, 'accel_time_scales': scales, 'disp_time_scales': scales, 'temporal_context_windows': [5, 15, 30], 'n_folds': 3, 'use_ensemble': False, 'xgb_rounds': 300, 'lgb_rounds': 300, 'early_stopping_rounds': 20, 'min_duration': 5, 'merge_gap': 10, 'smoothing_window': 5}

In [ ]:
# 数据读取与特征工程：生成 self/pair 行为识别所需的训练特征

train_meta_df = pl.read_csv(DATA_ROOT / 'train.csv')
excluded_video_ids = [1596473327, 878123481, 1212811043, 143861384]
train_meta_df = train_meta_df.filter(~pl.col('video_id').is_in(excluded_video_ids))
mouse4_fix_video_ids = [1351098077, 1260392287, 1643942986]

# 清理指定视频中与 mouse4 相关的行为标签，避免无效标注干扰训练。
def remove_mouse4_actions(behavior_json_text):
    try:
        behavior_items = json.loads(behavior_json_text)
        filtered_behavior_items = [item for item in behavior_items if 'mouse4' not in item]
        return json.dumps(filtered_behavior_items)
    except (json.JSONDecodeError, TypeError, AttributeError):
        return behavior_json_text
train_meta_df = train_meta_df.with_columns(pl.when(pl.col('video_id').is_in(mouse4_fix_video_ids)).then(pl.col('behaviors_labeled').map_elements(remove_mouse4_actions, return_dtype=pl.Utf8)).otherwise(pl.col('behaviors_labeled')).alias('behaviors_labeled'))
behavior_train_df = train_meta_df.filter(pl.col('behaviors_labeled').is_not_null()).select(pl.col('lab_id'), pl.col('video_id'), pl.col('behaviors_labeled').map_elements(eval, return_dtype=pl.List(pl.Utf8)).alias('behaviors_labeled_list')).explode('behaviors_labeled_list').rename({'behaviors_labeled_list': 'behaviors_labeled_element'}).select(pl.col('lab_id'), pl.col('video_id'), pl.col('behaviors_labeled_element').str.split(',').list[0].str.replace_all("'", '').alias('agent'), pl.col('behaviors_labeled_element').str.split(',').list[1].str.replace_all("'", '').alias('target'), pl.col('behaviors_labeled_element').str.split(',').list[2].str.replace_all("'", '').alias('behavior'))
self_behavior_train_df = behavior_train_df.filter(pl.col('behavior').is_in(SELF_ACTIONS))
pair_behavior_train_df = behavior_train_df.filter(pl.col('behavior').is_in(PAIR_ACTIONS))

# 构造单只小鼠的姿态、速度、角度和时间上下文特征。
def build_self_mouse_features(video_meta: dict, pose_df: pl.DataFrame) -> pl.DataFrame:
    """
    增强版self特征:
    - 多时间尺度速度
    - 加速度特征
    - 位移特征
    - 身体角度变化率
    - 伸长度变化率
    - 新增: 姿态特征、运动模式特征、静止检测
    """

    def add_part_distance(keypoint_a, keypoint_b):
        return ((pl.col(f'agent_x_{keypoint_a}') - pl.col(f'agent_x_{keypoint_b}')).pow(2) + (pl.col(f'agent_y_{keypoint_a}') - pl.col(f'agent_y_{keypoint_b}')).pow(2)).sqrt() / video_meta['pix_per_cm_approx']

    def add_part_speed(keypoint_name, time_lag_ms):
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        return ((pl.col(f'agent_x_{keypoint_name}').diff().pow(2) + pl.col(f'agent_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']).rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_displacement(keypoint_name, time_lag_ms):
        """一段时间内的总位移"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        return ((pl.col(f'agent_x_{keypoint_name}') - pl.col(f'agent_x_{keypoint_name}').shift(frame_window)).pow(2) + (pl.col(f'agent_y_{keypoint_name}') - pl.col(f'agent_y_{keypoint_name}').shift(frame_window)).pow(2)).sqrt() / video_meta['pix_per_cm_approx']

    def add_body_elongation():
        distance_left = add_part_distance('nose', 'tail_base')
        distance_right = add_part_distance('ear_left', 'ear_right')
        return distance_left / (distance_right + 1e-06)

    def add_body_angle():
        v1x = pl.col('agent_x_nose') - pl.col('agent_x_body_center')
        v1y = pl.col('agent_y_nose') - pl.col('agent_y_body_center')
        v2x = pl.col('agent_x_tail_base') - pl.col('agent_x_body_center')
        v2y = pl.col('agent_y_tail_base') - pl.col('agent_y_body_center')
        return (v1x * v2x + v1y * v2y) / ((v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt() + 1e-06)

    def add_acceleration(keypoint_name, time_lag_ms):
        """加速度: 速度的变化率"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        speed_values = (pl.col(f'agent_x_{keypoint_name}').diff().pow(2) + pl.col(f'agent_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']
        return speed_values.diff().rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_speed_std(keypoint_name, time_lag_ms):
        """速度的标准差 - 捕捉运动的不稳定性"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        speed_values = (pl.col(f'agent_x_{keypoint_name}').diff().pow(2) + pl.col(f'agent_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']
        return speed_values.rolling_std(window_size=frame_window, center=True, min_samples=1)

    def add_height_proxy(keypoint_name):
        """
        身体部位的'高度'代理 - 用y坐标的相对位置
        对于rear（站立）行为，nose的y坐标会相对较小（图像坐标系）
        """
        return pl.col(f'agent_y_{keypoint_name}') - pl.col('agent_y_body_center')

    def add_vertical_extension():
        """
        垂直伸展度: nose和tail_base的y坐标差
        rear时这个值会变大（nose向上）
        """
        return (pl.col('agent_y_tail_base') - pl.col('agent_y_nose')) / video_meta['pix_per_cm_approx']

    def add_body_compactness():
        """
        身体紧凑度: 所有部位到body_center的平均距离
        huddle/freeze时身体会更紧凑
        """
        mouse_parts = ['nose', 'ear_left', 'ear_right', 'tail_base', 'hip_left', 'hip_right']
        distance_list = []
        for keypoint_part in mouse_parts:
            distance_values = ((pl.col(f'agent_x_{keypoint_part}') - pl.col('agent_x_body_center')).pow(2) + (pl.col(f'agent_y_{keypoint_part}') - pl.col('agent_y_body_center')).pow(2)).sqrt() / video_meta['pix_per_cm_approx']
            distance_list.append(distance_values)
        return sum(distance_list) / len(distance_list)

    def add_tail_curvature():
        """
        尾巴弯曲度: tail_base到tail_tip的距离与body_center到tail_tip的距离之比
        """
        distance_left = add_part_distance('tail_base', 'tail_tip')
        distance_right = add_part_distance('body_center', 'tail_tip')
        return distance_left / (distance_right + 1e-06)

    def add_head_body_angle():
        """
        头部相对身体的角度
        用于检测grooming（头转向身体）
        """
        ear_mid_x = (pl.col('agent_x_ear_left') + pl.col('agent_x_ear_right')) / 2
        ear_mid_y = (pl.col('agent_y_ear_left') + pl.col('agent_y_ear_right')) / 2
        head_dir_x = pl.col('agent_x_nose') - ear_mid_x
        head_dir_y = pl.col('agent_y_nose') - ear_mid_y
        body_dir_x = pl.col('agent_x_tail_base') - pl.col('agent_x_body_center')
        body_dir_y = pl.col('agent_y_tail_base') - pl.col('agent_y_body_center')
        dot_product = head_dir_x * body_dir_x + head_dir_y * body_dir_y
        vector_norm_a = (head_dir_x.pow(2) + head_dir_y.pow(2)).sqrt()
        vector_norm_b = (body_dir_x.pow(2) + body_dir_y.pow(2)).sqrt()
        return dot_product / (vector_norm_a * vector_norm_b + 1e-06)

    def add_nose_body_distance():
        """
        nose到身体各部位的距离 - 用于检测selfgroom
        """
        return add_part_distance('nose', 'hip_left') + add_part_distance('nose', 'hip_right')

    def add_angular_velocity(time_lag_ms):
        """
        身体朝向的角速度
        用于检测旋转/转向行为
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        dir_x = pl.col('agent_x_nose') - pl.col('agent_x_tail_base')
        dir_y = pl.col('agent_y_nose') - pl.col('agent_y_tail_base')
        angle = pl.arctan2(dir_y, dir_x)
        return angle.diff().rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_movement_regularity(keypoint_name, time_lag_ms):
        """
        运动规律性: 位移与路径长度的比值
        值接近1表示直线运动，接近0表示原地打转
        用于区分run vs dig
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        net_disp = add_displacement(keypoint_name, time_lag_ms)
        frame_disp = (pl.col(f'agent_x_{keypoint_name}').diff().pow(2) + pl.col(f'agent_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx']
        path_length = frame_disp.rolling_sum(window_size=frame_window, center=True, min_samples=1)
        return net_disp / (path_length + 1e-06)

    def add_stillness_ratio(time_lag_ms, still_speed_threshold=1.0):
        """
        静止比例: 在时间窗口内速度低于阈值的帧数占比
        用于检测freeze, rest
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        speed_values = (pl.col('agent_x_body_center').diff().pow(2) + pl.col('agent_y_body_center').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']
        is_still = (speed_values < still_speed_threshold).cast(pl.Float32)
        return is_still.rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_limb_asymmetry():
        """
        肢体不对称性: 左右hip/lateral的距离差
        可能与特定行为姿态相关
        """
        left_dist = add_part_distance('body_center', 'hip_left')
        right_dist = add_part_distance('body_center', 'hip_right')
        return (left_dist - right_dist).abs()

    def add_elongation_change_rate(time_lag_ms):
        """伸长度变化率 - 捕捉身体伸缩"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        elong = add_body_elongation()
        return elong.diff().rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_body_angle_change_rate(time_lag_ms):
        """身体弯曲角度变化率"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        angle = add_body_angle()
        return angle.diff().rolling_mean(window_size=frame_window, center=True, min_samples=1)
    mouse_count = (video_meta['mouse1_strain'] is not None) + (video_meta['mouse2_strain'] is not None) + (video_meta['mouse3_strain'] is not None) + (video_meta['mouse4_strain'] is not None)
    start_frame_id = pose_df.select(pl.col('video_frame').min()).item()
    end_frame_id = pose_df.select(pl.col('video_frame').max()).item()
    result_df = []
    wide_pose_df = pose_df.pivot(on=['bodypart'], index=['video_frame', 'mouse_id'], values=['x', 'y']).sort(['mouse_id', 'video_frame'])
    wide_pose_tracks = {mouse_id: wide_pose_df.filter(pl.col('mouse_id') == mouse_id) for mouse_id in range(1, mouse_count + 1)}
    speed_keypoints = ['ear_left', 'ear_right', 'tail_base', 'body_center', 'nose']
    for agent_id in range(1, mouse_count + 1):
        segment_record = pl.DataFrame({'video_id': video_meta['video_id'], 'agent_mouse_id': agent_id, 'target_mouse_id': -1, 'video_frame': pl.arange(start_frame_id, end_frame_id + 1, eager=True)}, schema={'video_id': pl.Int32, 'agent_mouse_id': pl.Int8, 'target_mouse_id': pl.Int8, 'video_frame': pl.Int32})
        pivot_data = wide_pose_tracks[agent_id].select(pl.col('video_frame'), pl.exclude('video_frame').name.prefix('agent_'))
        available_columns = pivot_data.columns
        pivot_data = pivot_data.with_columns(*[pl.lit(None).cast(pl.Float32).alias(f'agent_x_{keypoint}') for keypoint in KEYPOINT_NAMES if f'agent_x_{keypoint}' not in available_columns], *[pl.lit(None).cast(pl.Float32).alias(f'agent_y_{keypoint}') for keypoint in KEYPOINT_NAMES if f'agent_y_{keypoint}' not in available_columns])
        feature_expressions = [pl.col('video_frame'), pl.lit(agent_id).alias('agent_mouse_id'), pl.lit(-1).alias('target_mouse_id')]
        feature_expressions.extend([add_part_distance(keypoint_a, keypoint_b).alias(f'aa__{keypoint_a}__{keypoint_b}__distance') for keypoint_a, keypoint_b in itertools.combinations(KEYPOINT_NAMES, 2)])
        feature_expressions.extend([add_part_speed(keypoint_name, time_lag_ms).alias(f'agent__{keypoint_name}__speed_{time_lag_ms}ms') for keypoint_name, time_lag_ms in itertools.product(speed_keypoints, MODEL_CONFIG['speed_time_scales'])])
        feature_expressions.extend([add_displacement('body_center', time_lag_ms).alias(f'agent__body_center__disp_{time_lag_ms}ms') for time_lag_ms in MODEL_CONFIG['disp_time_scales']])
        feature_expressions.append(add_body_elongation().alias('agent__elongation'))
        feature_expressions.append(add_body_angle().alias('agent__body_angle'))
        for time_lag_ms in scales:
            feature_expressions.append(add_acceleration('body_center', time_lag_ms).alias(f'agent__body_center__accel_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_speed_std('body_center', time_lag_ms).alias(f'agent__body_center__speed_std_{time_lag_ms}ms'))
        feature_expressions.append(add_vertical_extension().alias('agent__vertical_extension'))
        feature_expressions.append(add_body_compactness().alias('agent__body_compactness'))
        feature_expressions.append(add_tail_curvature().alias('agent__tail_curvature'))
        feature_expressions.append(add_head_body_angle().alias('agent__head_body_angle'))
        feature_expressions.append(add_nose_body_distance().alias('agent__nose_to_body_dist'))
        for time_lag_ms in scales:
            feature_expressions.append(add_angular_velocity(time_lag_ms).alias(f'agent__angular_velocity_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_movement_regularity('body_center', time_lag_ms).alias(f'agent__movement_regularity_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_stillness_ratio(time_lag_ms, still_speed_threshold=1.0).alias(f'agent__stillness_ratio_{time_lag_ms}ms'))
        feature_expressions.append(add_limb_asymmetry().alias('agent__limb_asymmetry'))
        for time_lag_ms in scales:
            feature_expressions.append(add_elongation_change_rate(time_lag_ms).alias(f'agent__elongation_change_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_body_angle_change_rate(time_lag_ms).alias(f'agent__body_angle_change_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            nose_speed = add_part_speed('nose', time_lag_ms)
            body_speed = add_part_speed('body_center', time_lag_ms)
            feature_expressions.append((nose_speed / (body_speed + 1e-06)).alias(f'agent__nose_body_speed_ratio_{time_lag_ms}ms'))
            tail_speed = add_part_speed('tail_base', time_lag_ms)
            feature_expressions.append((tail_speed / (body_speed + 1e-06)).alias(f'agent__tail_body_speed_ratio_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
            dx = pl.col('agent_x_body_center').diff()
            dy = pl.col('agent_y_body_center').diff()
            direction_change = dx * dx.shift(1) + dy * dy.shift(1) < 0
            feature_expressions.append(direction_change.cast(pl.Float32).rolling_mean(window_size=frame_window, center=True, min_samples=1).alias(f'agent__direction_reversal_ratio_{time_lag_ms}ms'))
        feature_columns = pivot_data.with_columns(pl.lit(agent_id).alias('agent_mouse_id'), pl.lit(-1).alias('target_mouse_id')).select(feature_expressions)
        segment_record = segment_record.join(feature_columns, on=['video_frame', 'agent_mouse_id', 'target_mouse_id'], how='left')
        result_df.append(segment_record)
    return pl.concat(result_df, how='vertical')

# 构造两只小鼠之间的距离、相对运动和交互方向特征。
def build_pair_mouse_features(video_meta: dict, pose_df: pl.DataFrame) -> pl.DataFrame:
    """
    增强版pair特征:
    - agent-target部位间距离
    - 多时间尺度速度
    - 相对速度
    - 接近/远离指标
    - 距离变化率 (新增)
    """

    def add_part_distance(mouse_role_a, keypoint_a, mouse_role_b, keypoint_b):
        return ((pl.col(f'{mouse_role_a}_x_{keypoint_a}') - pl.col(f'{mouse_role_b}_x_{keypoint_b}')).pow(2) + (pl.col(f'{mouse_role_a}_y_{keypoint_a}') - pl.col(f'{mouse_role_b}_y_{keypoint_b}')).pow(2)).sqrt() / video_meta['pix_per_cm_approx']

    def add_part_speed(mouse_role, keypoint_name, time_lag_ms):
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        return ((pl.col(f'{mouse_role}_x_{keypoint_name}').diff().pow(2) + pl.col(f'{mouse_role}_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']).rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_body_elongation(mouse_role):
        distance_left = add_part_distance(mouse_role, 'nose', mouse_role, 'tail_base')
        distance_right = add_part_distance(mouse_role, 'ear_left', mouse_role, 'ear_right')
        return distance_left / (distance_right + 1e-06)

    def add_body_angle(mouse_role):
        v1x = pl.col(f'{mouse_role}_x_nose') - pl.col(f'{mouse_role}_x_body_center')
        v1y = pl.col(f'{mouse_role}_y_nose') - pl.col(f'{mouse_role}_y_body_center')
        v2x = pl.col(f'{mouse_role}_x_tail_base') - pl.col(f'{mouse_role}_x_body_center')
        v2y = pl.col(f'{mouse_role}_y_tail_base') - pl.col(f'{mouse_role}_y_body_center')
        return (v1x * v2x + v1y * v2y) / ((v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt() + 1e-06)

    def add_center_distance_rolling(agg, time_lag_ms):
        expr = add_part_distance('agent', 'body_center', 'target', 'body_center')
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        if agg == 'mean':
            return expr.rolling_mean(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'std':
            return expr.rolling_std(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'min':
            return expr.rolling_min(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'max':
            return expr.rolling_max(window_size=frame_window, center=True, min_samples=1)
        else:
            raise ValueError()

    def add_distance_change_rate(keypoint_a, keypoint_b, time_lag_ms):
        """
        计算两个部位之间距离的变化率
        正值 = 远离, 负值 = 接近
        单位: cm/s
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        distance_values = add_part_distance('agent', keypoint_a, 'target', keypoint_b)
        dist_change = (distance_values - distance_values.shift(frame_window)) / (frame_window / video_meta['frames_per_second'])
        return dist_change

    def add_distance_difference(keypoint_a, keypoint_b):
        """
        帧间距离变化 (用于后续rolling计算)
        """
        distance_values = add_part_distance('agent', keypoint_a, 'target', keypoint_b)
        return distance_values.diff() * video_meta['frames_per_second']

    def add_distance_change_rolling(keypoint_a, keypoint_b, time_lag_ms, agg='mean'):
        """
        距离变化的滚动统计
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        dist_diff = add_distance_difference(keypoint_a, keypoint_b)
        if agg == 'mean':
            return dist_diff.rolling_mean(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'std':
            return dist_diff.rolling_std(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'min':
            return dist_diff.rolling_min(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'max':
            return dist_diff.rolling_max(window_size=frame_window, center=True, min_samples=1)
        else:
            raise ValueError()

    def add_approaching_ratio(keypoint_a, keypoint_b, time_lag_ms):
        """
        计算在一段时间窗口内，接近的帧数占比
        值域 [0, 1], 接近1表示一直在接近
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        distance_values = add_part_distance('agent', keypoint_a, 'target', keypoint_b)
        is_approaching = (distance_values.diff() < 0).cast(pl.Float32)
        return is_approaching.rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_agent_facing_target():
        """agent是否面向target"""
        agent_dir_x = pl.col('agent_x_nose') - pl.col('agent_x_body_center')
        agent_dir_y = pl.col('agent_y_nose') - pl.col('agent_y_body_center')
        to_target_x = pl.col('target_x_body_center') - pl.col('agent_x_body_center')
        to_target_y = pl.col('target_y_body_center') - pl.col('agent_y_body_center')
        dot_product = agent_dir_x * to_target_x + agent_dir_y * to_target_y
        vector_norm_a = (agent_dir_x.pow(2) + agent_dir_y.pow(2)).sqrt()
        vector_norm_b = (to_target_x.pow(2) + to_target_y.pow(2)).sqrt()
        return dot_product / (vector_norm_a * vector_norm_b + 1e-06)

    def add_target_facing_agent():
        """target是否面向agent"""
        target_dir_x = pl.col('target_x_nose') - pl.col('target_x_body_center')
        target_dir_y = pl.col('target_y_nose') - pl.col('target_y_body_center')
        to_agent_x = pl.col('agent_x_body_center') - pl.col('target_x_body_center')
        to_agent_y = pl.col('agent_y_body_center') - pl.col('target_y_body_center')
        dot_product = target_dir_x * to_agent_x + target_dir_y * to_agent_y
        vector_norm_a = (target_dir_x.pow(2) + target_dir_y.pow(2)).sqrt()
        vector_norm_b = (to_agent_x.pow(2) + to_agent_y.pow(2)).sqrt()
        return dot_product / (vector_norm_a * vector_norm_b + 1e-06)

    def add_agent_behind_target():
        """agent是否在target后面"""
        target_dir_x = pl.col('target_x_nose') - pl.col('target_x_tail_base')
        target_dir_y = pl.col('target_y_nose') - pl.col('target_y_tail_base')
        to_agent_x = pl.col('agent_x_body_center') - pl.col('target_x_body_center')
        to_agent_y = pl.col('agent_y_body_center') - pl.col('target_y_body_center')
        dot_product = target_dir_x * to_agent_x + target_dir_y * to_agent_y
        vector_norm_a = (target_dir_x.pow(2) + target_dir_y.pow(2)).sqrt()
        vector_norm_b = (to_agent_x.pow(2) + to_agent_y.pow(2)).sqrt()
        return -dot_product / (vector_norm_a * vector_norm_b + 1e-06)

    def add_agent_nose_to_rear():
        """agent的nose是否更靠近target的后部"""
        nose_to_tail = add_part_distance('agent', 'nose', 'target', 'tail_base')
        nose_to_nose = add_part_distance('agent', 'nose', 'target', 'nose')
        return nose_to_nose / (nose_to_tail + 1e-06)

    def add_relative_speed(time_lag_ms):
        """相对速度"""
        agent_speed = add_part_speed('agent', 'body_center', time_lag_ms)
        target_speed = add_part_speed('target', 'body_center', time_lag_ms)
        return agent_speed - target_speed

    def add_speed_ratio(time_lag_ms):
        """速度比值"""
        agent_speed = add_part_speed('agent', 'body_center', time_lag_ms)
        target_speed = add_part_speed('target', 'body_center', time_lag_ms)
        return agent_speed / (target_speed + 1e-06)
    mouse_count = (video_meta['mouse1_strain'] is not None) + (video_meta['mouse2_strain'] is not None) + (video_meta['mouse3_strain'] is not None) + (video_meta['mouse4_strain'] is not None)
    start_frame_id = pose_df.select(pl.col('video_frame').min()).item()
    end_frame_id = pose_df.select(pl.col('video_frame').max()).item()
    result_df = []
    wide_pose_df = pose_df.pivot(on=['bodypart'], index=['video_frame', 'mouse_id'], values=['x', 'y']).sort(['mouse_id', 'video_frame'])
    wide_pose_tracks = {mouse_id: wide_pose_df.filter(pl.col('mouse_id') == mouse_id) for mouse_id in range(1, mouse_count + 1)}
    speed_keypoints = ['ear_left', 'ear_right', 'tail_base', 'body_center']
    dist_change_time_scales = scales
    for agent_id, target_id in itertools.permutations(range(1, mouse_count + 1), 2):
        segment_record = pl.DataFrame({'video_id': video_meta['video_id'], 'agent_mouse_id': agent_id, 'target_mouse_id': target_id, 'video_frame': pl.arange(start_frame_id, end_frame_id + 1, eager=True)}, schema={'video_id': pl.Int32, 'agent_mouse_id': pl.Int8, 'target_mouse_id': pl.Int8, 'video_frame': pl.Int32})
        merged_pivot = wide_pose_tracks[agent_id].select(pl.col('video_frame'), pl.exclude('video_frame').name.prefix('agent_')).join(wide_pose_tracks[target_id].select(pl.col('video_frame'), pl.exclude('video_frame').name.prefix('target_')), on='video_frame', how='inner')
        available_columns = merged_pivot.columns
        merged_pivot = merged_pivot.with_columns(*[pl.lit(None).cast(pl.Float32).alias(f'agent_x_{keypoint}') for keypoint in KEYPOINT_NAMES if f'agent_x_{keypoint}' not in available_columns], *[pl.lit(None).cast(pl.Float32).alias(f'agent_y_{keypoint}') for keypoint in KEYPOINT_NAMES if f'agent_y_{keypoint}' not in available_columns], *[pl.lit(None).cast(pl.Float32).alias(f'target_x_{keypoint}') for keypoint in KEYPOINT_NAMES if f'target_x_{keypoint}' not in available_columns], *[pl.lit(None).cast(pl.Float32).alias(f'target_y_{keypoint}') for keypoint in KEYPOINT_NAMES if f'target_y_{keypoint}' not in available_columns])
        feature_expressions = [pl.col('video_frame'), pl.lit(agent_id).alias('agent_mouse_id'), pl.lit(target_id).alias('target_mouse_id')]
        feature_expressions.extend([add_part_distance('agent', agent_body_part, 'target', target_body_part).alias(f'at__{agent_body_part}__{target_body_part}__distance') for agent_body_part, target_body_part in itertools.product(KEYPOINT_NAMES, repeat=2)])
        feature_expressions.extend([add_part_speed('agent', keypoint_name, time_lag_ms).alias(f'agent__{keypoint_name}__speed_{time_lag_ms}ms') for keypoint_name, time_lag_ms in itertools.product(speed_keypoints, MODEL_CONFIG['speed_time_scales'])])
        feature_expressions.extend([add_part_speed('target', keypoint_name, time_lag_ms).alias(f'target__{keypoint_name}__speed_{time_lag_ms}ms') for keypoint_name, time_lag_ms in itertools.product(speed_keypoints, MODEL_CONFIG['speed_time_scales'])])
        feature_expressions.extend([add_body_elongation('agent').alias('agent__elongation'), add_body_elongation('target').alias('target__elongation'), add_body_angle('agent').alias('agent__body_angle'), add_body_angle('target').alias('target__body_angle')])
        for agg in ['mean', 'std', 'min', 'max']:
            for time_lag_ms in scales:
                feature_expressions.append(add_center_distance_rolling(agg, time_lag_ms).alias(f'at__body_center__dist_{agg}_{time_lag_ms}ms'))
        for time_lag_ms in dist_change_time_scales:
            feature_expressions.append(add_distance_change_rate('body_center', 'body_center', time_lag_ms).alias(f'at__body_center__dist_change_rate_{time_lag_ms}ms'))
        key_target_parts = ['nose', 'body_center', 'tail_base']
        for target_part in key_target_parts:
            for time_lag_ms in scales:
                feature_expressions.append(add_distance_change_rate('nose', target_part, time_lag_ms).alias(f'at__nose__{target_part}__dist_change_rate_{time_lag_ms}ms'))
        for agg in ['mean', 'std']:
            for time_lag_ms in scales:
                feature_expressions.append(add_distance_change_rolling('body_center', 'body_center', time_lag_ms, agg).alias(f'at__body_center__dist_change_{agg}_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_approaching_ratio('body_center', 'body_center', time_lag_ms).alias(f'at__body_center__approaching_ratio_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_distance_change_rolling('body_center', 'body_center', time_lag_ms, 'min').alias(f'at__body_center__dist_change_min_{time_lag_ms}ms'))
            feature_expressions.append(add_distance_change_rolling('body_center', 'body_center', time_lag_ms, 'max').alias(f'at__body_center__dist_change_max_{time_lag_ms}ms'))
        feature_expressions.extend([add_agent_facing_target().alias('at__agent_facing_target'), add_target_facing_agent().alias('at__target_facing_agent'), (add_agent_facing_target() * add_target_facing_agent()).alias('at__mutual_facing'), add_agent_behind_target().alias('at__agent_behind_target'), add_agent_nose_to_rear().alias('at__agent_nose_to_target_rear')])
        for time_lag_ms in scales:
            feature_expressions.extend([add_relative_speed(time_lag_ms).alias(f'at__relative_speed_{time_lag_ms}ms'), add_speed_ratio(time_lag_ms).alias(f'at__speed_ratio_{time_lag_ms}ms')])
        feature_columns = merged_pivot.with_columns(pl.lit(agent_id).alias('agent_mouse_id'), pl.lit(target_id).alias('target_mouse_id')).select(feature_expressions)
        segment_record = segment_record.join(feature_columns, on=['video_frame', 'agent_mouse_id', 'target_mouse_id'], how='left')
        result_df.append(segment_record)
    return pl.concat(result_df, how='vertical')

# 为帧级特征追加前后时刻的统计信息，增强时序表达。
def append_temporal_context(features_df: pl.DataFrame, windows: list=None) -> pl.DataFrame:
    """
    为特征添加时序上下文: rolling mean/std
    """
    if windows is None:
        windows = MODEL_CONFIG['temporal_context_windows']
    feature_cols = [c for c in features_df.columns if c not in KEY_COLUMNS]
    important_keywords = ['speed_500ms', 'speed_1000ms', 'distance', 'elongation', 'angle']
    important_features = [c for c in feature_cols if any((kw in c for kw in important_keywords))][:15]
    new_cols = []
    for window_size in windows:
        for col in important_features:
            new_cols.extend([pl.col(col).rolling_mean(window_size=window_size, min_samples=1).alias(f'{col}_rmean{window_size}'), pl.col(col).rolling_std(window_size=window_size, min_samples=1).alias(f'{col}_rstd{window_size}')])
    if new_cols:
        return features_df.with_columns(new_cols)
    return features_df

# 对逐帧预测概率进行时间平滑，降低帧级抖动。
def smooth_time_probabilities(raw_predictions: np.ndarray, decision_threshold: float, min_segment_length: int=None, smooth_window_size: int=None) -> np.ndarray:
    """
    时序平滑: 移动平均 + 短片段过滤
    返回平滑后的二值预测
    """
    if min_segment_length is None:
        min_segment_length = MODEL_CONFIG['min_duration']
    if smooth_window_size is None:
        smooth_window_size = MODEL_CONFIG['smoothing_window']
    if smooth_window_size > 1:
        smooth_kernel = np.ones(smooth_window_size) / smooth_window_size
        smoothed = np.convolve(raw_predictions, smooth_kernel, mode='same')
    else:
        smoothed = raw_predictions
    binary = (smoothed >= decision_threshold).astype(np.int8)
    if min_segment_length > 1:
        diff = np.diff(np.concatenate([[0], binary, [0]]))
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]
        for start, end in zip(starts, ends):
            if end - start < min_segment_length:
                binary[start:end] = 0
    return binary

# 在验证集上搜索使 F1 分数最高的分类阈值。
def search_best_threshold(oof_probabilities: np.ndarray, true_values: np.ndarray, enable_smoothing: bool=True) -> float:
    """
    调优阈值, 可选择是否结合后处理
    """
    fold_thresholds = np.arange(0.1, 0.9, 0.01)
    best_f1_value = 0
    best_threshold_value = 0.5
    for th in fold_thresholds:
        if enable_smoothing:
            predicted_labels = smooth_time_probabilities(oof_probabilities, th)
        else:
            predicted_labels = (oof_probabilities >= th).astype(int)
        f1_value = f1_score(true_values, predicted_labels, zero_division=0)
        if f1_value > best_f1_value:
            best_f1_value = f1_value
            best_threshold_value = th
    return best_threshold_value

# 训练单个行为模型，并返回验证预测与最优阈值。
def fit_validate_action_model(lab_name: str, action_name: str, sample_indices: pl.DataFrame, feature_columns: pl.DataFrame, target_values: pl.Series):
    """
    训练和验证函数
    支持XGBoost + LightGBM集成
    """
    result_dir = ARTIFACT_DIR / 'results' / lab_name / action_name
    result_dir.mkdir(exist_ok=True, parents=True)
    if target_values.sum() == 0:
        with open(result_dir / 'f1.txt', 'w') as f:
            f.write('0.0\n')
        oof_prediction_df = sample_indices.with_columns(pl.Series('fold', [-1] * len(target_values), dtype=pl.Int8), pl.Series('prediction', [0.0] * len(target_values), dtype=pl.Float32), pl.Series('predicted_label', [0] * len(target_values), dtype=pl.Int8))
        oof_prediction_df.write_parquet(result_dir / 'oof_predictions.parquet')
        return 0.0
    feature_matrix = feature_columns.to_numpy()
    y = target_values.to_numpy()
    model_feature_names = feature_columns.columns
    folds = np.ones(len(y), dtype=np.int8) * -1
    oof_probabilities = np.zeros(len(y), dtype=np.float32)
    oof_prediction_labels = np.zeros(len(y), dtype=np.int8)
    n_folds = MODEL_CONFIG['n_folds']
    for fold_id, (train_idx, valid_idx) in enumerate(StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=42).split(X=feature_matrix, y=y, groups=sample_indices.get_column('video_id').to_numpy())):
        result_dir_fold = result_dir / f'fold_{fold_id}'
        result_dir_fold.mkdir(exist_ok=True, parents=True)
        X_train, X_valid = (feature_matrix[train_idx], feature_matrix[valid_idx])
        y_train, y_valid = (y[train_idx], y[valid_idx])
        scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
        xgb_params = {'objective': 'binary:logistic', 'eval_metric': 'logloss', 'device': 'cpu', 'tree_method': 'hist', 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'scale_pos_weight': scale_pos_weight, 'max_bin': 64, 'seed': 42}
        dtrain = xgb.QuantileDMatrix(X_train, label=y_train, feature_names=model_feature_names, max_bin=64)
        dvalid = xgb.DMatrix(X_valid, label=y_valid, feature_names=model_feature_names)
        early_stopping_callback = xgb.callback.EarlyStopping(rounds=MODEL_CONFIG['early_stopping_rounds'], metric_name='logloss', data_name='valid', maximize=False, save_best=True)
        xgb_model = xgb.train(xgb_params, dtrain=dtrain, num_boost_round=MODEL_CONFIG['xgb_rounds'], evals=[(dtrain, 'train'), (dvalid, 'valid')], callbacks=[early_stopping_callback], verbose_eval=0)
        xgb_pred = xgb_model.predict(dvalid)
        xgb_model.save_model(result_dir_fold / 'xgb_model.json')
        if MODEL_CONFIG['use_ensemble']:
            lgb_params = {'objective': 'binary', 'metric': 'binary_logloss', 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'scale_pos_weight': scale_pos_weight, 'seed': 42, 'verbose': -1, 'force_col_wise': True}
            lgb_train = lgb.Dataset(X_train, y_train, feature_name=model_feature_names)
            lgb_valid = lgb.Dataset(X_valid, y_valid, feature_name=model_feature_names, reference=lgb_train)
            lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=MODEL_CONFIG['lgb_rounds'], valid_sets=[lgb_valid], callbacks=[lgb.early_stopping(MODEL_CONFIG['early_stopping_rounds'], verbose=False), lgb.log_evaluation(period=0)])
            lgb_pred = lgb_model.predict(X_valid)
            lgb_model.save_model(str(result_dir_fold / 'lgb_model.txt'))
            fold_predictions = (xgb_pred + lgb_pred) / 2
        else:
            fold_predictions = xgb_pred
        decision_threshold = search_best_threshold(fold_predictions, y_valid, enable_smoothing=True)
        predicted_labels = smooth_time_probabilities(fold_predictions, decision_threshold)
        if MODEL_CONFIG['use_ensemble']:
            dtrain_pred = xgb.DMatrix(X_train, feature_names=model_feature_names)
            train_preds = (xgb_model.predict(dtrain_pred) + lgb_model.predict(X_train)) / 2
        else:
            dtrain_pred = xgb.DMatrix(X_train, feature_names=model_feature_names)
            train_preds = xgb_model.predict(dtrain_pred)
        train_labels = smooth_time_probabilities(train_preds, decision_threshold)
        train_f1 = f1_score(y_train, train_labels, zero_division=0)
        valid_f1 = f1_score(y_valid, predicted_labels, zero_division=0)
        gap = train_f1 - valid_f1
        print(f'    Fold {fold_id}: Train F1={train_f1:.4f}, Valid F1={valid_f1:.4f}, Gap={gap:.4f}')
        folds[valid_idx] = fold_id
        oof_probabilities[valid_idx] = fold_predictions
        oof_prediction_labels[valid_idx] = predicted_labels
        with open(result_dir_fold / 'threshold.txt', 'w') as f:
            f.write(f'{decision_threshold}\n')
        xgb.plot_importance(xgb_model, max_num_features=20, importance_type='gain', values_format='{v:.2f}')
        plt.tight_layout()
        plt.savefig(result_dir_fold / 'feature_importance.png')
        plt.close()
        gc.collect()
    oof_prediction_df = sample_indices.with_columns(pl.Series('fold', folds, dtype=pl.Int8), pl.Series('prediction', oof_probabilities, dtype=pl.Float32), pl.Series('predicted_label', oof_prediction_labels, dtype=pl.Int8))
    f1_value = f1_score(y, oof_prediction_labels, zero_division=0)
    with open(result_dir / 'f1.txt', 'w') as f:
        f.write(f'{f1_value}\n')
    oof_prediction_df.write_parquet(result_dir / 'oof_predictions.parquet')
    return f1_value

# 按视频读取姿态与标注数据，生成 self/pair 两类训练样本。
def prepare_video_features(video_row):
    """处理单个视频，生成特征"""
    lab_name = video_row['lab_id']
    video_key = video_row['video_id']
    tracking_path = TRAIN_POSE_DIR / f'{lab_name}/{video_key}.parquet'
    pose_df = pl.read_parquet(tracking_path)
    self_features = build_self_mouse_features(video_meta=video_row, pose_df=pose_df)
    pair_features = build_pair_mouse_features(video_meta=video_row, pose_df=pose_df)
    self_features.write_parquet(ARTIFACT_DIR / 'self_features' / f'{video_key}.parquet')
    pair_features.write_parquet(ARTIFACT_DIR / 'pair_features' / f'{video_key}.parquet')
    return video_key

In [ ]:
# 批量生成特征文件：按视频并行处理，减少后续训练的重复计算

print('=' * 60)
print('开始生成特征...')
print('=' * 60)
(ARTIFACT_DIR / 'self_features').mkdir(exist_ok=True, parents=True)
(ARTIFACT_DIR / 'pair_features').mkdir(exist_ok=True, parents=True)
video_rows = list(train_meta_df.filter(pl.col('behaviors_labeled').is_not_null()).rows(named=True))
feature_build_results = joblib.Parallel(n_jobs=-1, verbose=5)((joblib.delayed(prepare_video_features)(video_row) for video_row in video_rows))
print(f'成功处理 {len(feature_build_results)} 个视频')
del video_rows, feature_build_results
gc.collect()

In [ ]:
# 训练 Self 行为模型：按实验室与动作分组训练二分类模型

print('\n' + '=' * 60)
print('开始训练 Self 行为模型...')
print('=' * 60)
behavior_groups = self_behavior_train_df.group_by('lab_id', 'behavior', maintain_order=True)
num_behavior_groups = len(list(behavior_groups))
phase_start_time = time.perf_counter()
self_cv_f1_scores = []
for group_idx, ((lab_name, action_name), behavior_group) in tqdm(enumerate(behavior_groups), total=num_behavior_groups):
    if group_idx == 0:
        tqdm.write(f"|{'LAB':^25}|{'BEHAVIOR':^15}|{'SAMPLES':^10}|{'POSITIVE':^10}|{'FEATURES':^10}|{'F1':^10}|{'ELAPSED TIME':^15}|", end='\n')
    tqdm.write(f'|{lab_name:^25}|{action_name:^15}|', end='')
    index_chunks = []
    feature_chunks = []
    label_chunks = []
    for video_row in behavior_group.rows(named=True):
        video_key = video_row['video_id']
        agent_role = video_row['agent']
        agent_id = int(re.search('mouse(\\d+)', agent_role).group(1))
        working_df = pl.scan_parquet(ARTIFACT_DIR / 'self_features' / f'{video_key}.parquet').filter(pl.col('agent_mouse_id') == agent_id)
        index = working_df.select(KEY_COLUMNS).collect(engine='streaming')
        feature_name = working_df.select(pl.exclude(KEY_COLUMNS)).collect(engine='streaming')
        annotation_file_path = TRAIN_LABEL_DIR / lab_name / f'{video_key}.parquet'
        if annotation_file_path.exists():
            annotation_df = pl.scan_parquet(annotation_file_path).filter((pl.col('action') == action_name) & (pl.col('agent_id') == agent_id)).collect()
        else:
            annotation_df = pl.DataFrame(schema={'agent_id': pl.Int8, 'target_id': pl.Int8, 'action': str, 'start_frame': pl.Int16, 'stop_frame': pl.Int16})
        positive_frame_ids = set()
        for annotation_row_df in annotation_df.rows(named=True):
            positive_frame_ids.update(range(annotation_row_df['start_frame'], annotation_row_df['stop_frame']))
        label_df = index.select(pl.col('video_frame').is_in(positive_frame_ids).cast(pl.Int8).alias('label'))
        if label_df.get_column('label').sum() == 0:
            continue
        index_chunks.append(index)
        feature_chunks.append(feature_name)
        label_chunks.append(label_df.get_column('label'))
    if not index_chunks:
        elapsed_seconds = datetime.timedelta(seconds=int(time.perf_counter() - phase_start_time))
        tqdm.write(f"{0:>10,}|{0:>10,}|{0:>10,}|{'-':>10}|{str(elapsed_seconds):>15}|", end='\n')
        continue
    sample_indices = pl.concat(index_chunks, how='vertical')
    feature_columns = pl.concat(feature_chunks, how='vertical')
    target_values = pl.concat(label_chunks, how='vertical')
    del index_chunks, feature_chunks, label_chunks
    gc.collect()
    tqdm.write(f'{len(sample_indices):>10,}|{target_values.sum():>10,}|{len(feature_columns.columns):>10,}|', end='')
    f1_value = fit_validate_action_model(lab_name, action_name, sample_indices, feature_columns, target_values)
    self_cv_f1_scores.append({'lab_id': lab_name, 'behavior': action_name, 'f1': f1_value})
    tqdm.write(f'{f1_value:>10.2f}|', end='')
    elapsed_seconds = datetime.timedelta(seconds=int(time.perf_counter() - phase_start_time))
    tqdm.write(f'{str(elapsed_seconds):>15}|', end='\n')
    gc.collect()
print(f"\nSelf行为平均F1: {np.mean([s['f1'] for s in self_cv_f1_scores if s['f1'] > 0]):.4f}")

In [ ]:
# 训练 Pair 行为模型：针对两只小鼠交互动作分别建模

print('\n' + '=' * 60)
print('开始训练 Pair 行为模型...')
print('=' * 60)
behavior_groups = pair_behavior_train_df.group_by('lab_id', 'behavior', maintain_order=True)
num_behavior_groups = len(list(behavior_groups))
phase_start_time = time.perf_counter()
pair_cv_f1_scores = []
for group_idx, ((lab_name, action_name), behavior_group) in tqdm(enumerate(behavior_groups), total=num_behavior_groups):
    if group_idx == 0:
        tqdm.write(f"|{'LAB':^25}|{'BEHAVIOR':^15}|{'SAMPLES':^10}|{'POSITIVE':^10}|{'FEATURES':^10}|{'F1':^10}|{'ELAPSED TIME':^15}|", end='\n')
    if lab_name == 'BoisterousParrot':
        continue
    tqdm.write(f'|{lab_name:^25}|{action_name:^15}|', end='')
    index_chunks = []
    feature_chunks = []
    label_chunks = []
    for video_row in behavior_group.rows(named=True):
        video_key = video_row['video_id']
        agent_role = video_row['agent']
        target_role = video_row['target']
        agent_id = int(re.search('mouse(\\d+)', agent_role).group(1))
        target_id = int(re.search('mouse(\\d+)', target_role).group(1))
        working_df = pl.scan_parquet(ARTIFACT_DIR / 'pair_features' / f'{video_key}.parquet').filter((pl.col('agent_mouse_id') == agent_id) & (pl.col('target_mouse_id') == target_id))
        index = working_df.select(KEY_COLUMNS).collect(engine='streaming')
        feature_name = working_df.select(pl.exclude(KEY_COLUMNS)).collect(engine='streaming')
        annotation_file_path = TRAIN_LABEL_DIR / lab_name / f'{video_key}.parquet'
        if annotation_file_path.exists():
            annotation_df = pl.scan_parquet(annotation_file_path).filter((pl.col('action') == action_name) & (pl.col('agent_id') == agent_id) & (pl.col('target_id') == target_id)).collect()
        else:
            annotation_df = pl.DataFrame(schema={'agent_id': pl.Int8, 'target_id': pl.Int8, 'action': str, 'start_frame': pl.Int16, 'stop_frame': pl.Int16})
        positive_frame_ids = set()
        for annotation_row_df in annotation_df.rows(named=True):
            positive_frame_ids.update(range(annotation_row_df['start_frame'], annotation_row_df['stop_frame']))
        label_df = index.select(pl.col('video_frame').is_in(positive_frame_ids).cast(pl.Int8).alias('label'))
        if label_df.get_column('label').sum() == 0:
            continue
        index_chunks.append(index)
        feature_chunks.append(feature_name)
        label_chunks.append(label_df.get_column('label'))
    if not index_chunks:
        elapsed_seconds = datetime.timedelta(seconds=int(time.perf_counter() - phase_start_time))
        tqdm.write(f"{0:>10,}|{0:>10,}|{0:>10,}|{'-':>10}|{str(elapsed_seconds):>15}|", end='\n')
        continue
    sample_indices = pl.concat(index_chunks, how='vertical')
    feature_columns = pl.concat(feature_chunks, how='vertical')
    target_values = pl.concat(label_chunks, how='vertical')
    del index_chunks, feature_chunks, label_chunks
    gc.collect()
    tqdm.write(f'{len(sample_indices):>10,}|{target_values.sum():>10,}|{len(feature_columns.columns):>10,}|', end='')
    f1_value = fit_validate_action_model(lab_name, action_name, sample_indices, feature_columns, target_values)
    pair_cv_f1_scores.append({'lab_id': lab_name, 'behavior': action_name, 'f1': f1_value})
    tqdm.write(f'{f1_value:>10.2f}|', end='')
    elapsed_seconds = datetime.timedelta(seconds=int(time.perf_counter() - phase_start_time))
    tqdm.write(f'{str(elapsed_seconds):>15}|', end='\n')
    gc.collect()
print(f"\nPair行为平均F1: {np.mean([s['f1'] for s in pair_cv_f1_scores if s['f1'] > 0]):.4f}")

In [ ]:
# 后处理函数：清理预测片段并提升提交结果的稳定性

# 对预测片段进行后处理，修正异常区间并过滤无效结果。
def clean_submission_segments(submission_df: pl.DataFrame, metadata_df: pl.DataFrame, train_test: str='train'):
    traintest_directory = DATA_ROOT / f'{train_test}_tracking'
    raw_submission_df = submission_df.clone()
    submission_df = submission_df.filter(pl.col('start_frame') < pl.col('stop_frame'))
    if len(submission_df) != len(raw_submission_df):
        print('WARNING: Dropped frames with start >= stop')
    raw_submission_df = submission_df.clone()
    group_list = []
    for _, behavior_group in submission_df.group_by('video_id', 'agent_id', 'target_id'):
        behavior_group = behavior_group.sort('start_frame')
        mask = np.ones(len(behavior_group), dtype=bool)
        last_stop_frame = 0
        for i, video_row in enumerate(behavior_group.rows(named=True)):
            if video_row['start_frame'] < last_stop_frame:
                mask[i] = False
            else:
                last_stop_frame = video_row['stop_frame']
        group_list.append(behavior_group.filter(pl.Series('mask', mask)))
    submission_df = pl.concat(group_list)
    if len(submission_df) != len(raw_submission_df):
        print('WARNING: Dropped duplicate frames')
    s_list = []
    for video_row in metadata_df.rows(named=True):
        lab_name = video_row['lab_id']
        video_key = video_row['video_id']
        if video_row['behaviors_labeled'] is None:
            continue
        if video_key in submission_df.get_column('video_id').to_list():
            continue
        if isinstance(video_row['behaviors_labeled'], str):
            continue
        print(f'Video {video_key} has no predictions.')
        file_path = traintest_directory / f'/{lab_name}/{video_key}.parquet'
        vid = pd.read_parquet(file_path)
        vid_behaviors = json.loads(video_row['behaviors_labeled'])
        vid_behaviors = sorted(list({b.replace("'", '') for b in vid_behaviors}))
        vid_behaviors = [b.split(',') for b in vid_behaviors]
        vid_behaviors = pd.DataFrame(vid_behaviors, columns=['agent', 'target', 'action'])
        start_frame_id = vid.video_frame.min()
        stop_frame = vid.video_frame.max() + 1
        for (agent_role, target_role), actions in vid_behaviors.groupby(['agent', 'target']):
            batch_length = int(np.ceil((stop_frame - start_frame_id) / len(actions)))
            for i, action_row in enumerate(actions.itertuples(index=False)):
                batch_start = start_frame_id + i * batch_length
                batch_stop = min(batch_start + batch_length, stop_frame)
                s_list.append((video_key, agent_role, target_role, action_row[2], batch_start, batch_stop))
    if len(s_list) > 0:
        submission_df = pl.concat([submission_df, pl.DataFrame(s_list, schema=['video_id', 'agent_id', 'target_id', 'action', 'start_frame', 'stop_frame'])])
        print('WARNING: Filled empty videos')
    return submission_df

In [ ]:
# OOF 聚合：把各行为模型的验证集预测合并成片段级结果

print('\n' + '=' * 60)
print('聚合OOF预测结果...')
print('=' * 60)
group_oof_prediction_tables = []
behavior_groups = behavior_train_df.group_by('lab_id', 'video_id', 'agent', 'target', maintain_order=True)
for (lab_name, video_key, agent_role, target_role), behavior_group in tqdm(behavior_groups, total=len(list(behavior_groups))):
    agent_id = int(re.search('mouse(\\d+)', agent_role).group(1))
    target_id = -1 if target_role == 'self' else int(re.search('mouse(\\d+)', target_role).group(1))
    prediction_dataframe_list = []
    for video_row in behavior_group.rows(named=True):
        action_name = video_row['behavior']
        oof_path = ARTIFACT_DIR / 'results' / lab_name / action_name / 'oof_predictions.parquet'
        if not oof_path.exists():
            continue
        prediction = pl.scan_parquet(oof_path).filter((pl.col('video_id') == video_key) & (pl.col('agent_mouse_id') == agent_id) & (pl.col('target_mouse_id') == target_id)).select(*KEY_COLUMNS, (pl.col('prediction') * pl.col('predicted_label')).alias(action_name)).collect()
        if len(prediction) == 0:
            continue
        prediction_dataframe_list.append(prediction)
    if not prediction_dataframe_list:
        continue
    prediction_dataframe = pl.concat(prediction_dataframe_list, how='align')
    cols = prediction_dataframe.select(pl.exclude(KEY_COLUMNS)).columns
    prediction_labels_dataframe = prediction_dataframe.with_columns(pl.struct(pl.exclude(KEY_COLUMNS)).map_elements(lambda video_row: 'none' if sum(video_row.values()) == 0 else cols[np.argmax(list(video_row.values()))], return_dtype=pl.String).alias('prediction')).select(KEY_COLUMNS + ['prediction'])
    group_oof_prediction = prediction_labels_dataframe.filter(pl.col('prediction') != pl.col('prediction').shift(1)).with_columns(pl.col('video_frame').shift(-1).alias('stop_frame')).filter(pl.col('prediction') != 'none').select(pl.col('video_id'), ('mouse' + pl.col('agent_mouse_id').cast(str)).alias('agent_id'), pl.when(pl.col('target_mouse_id') == -1).then(pl.lit('self')).otherwise('mouse' + pl.col('target_mouse_id').cast(str)).alias('target_id'), pl.col('prediction').alias('action'), pl.col('video_frame').alias('start_frame'), pl.col('stop_frame'))
    group_oof_prediction_tables.append(group_oof_prediction)
oof_probabilities = pl.concat(group_oof_prediction_tables, how='vertical')
oof_probabilities = clean_submission_segments(oof_probabilities, train_meta_df, train_test='train')
oof_probabilities.with_row_index('row_id').write_csv(ARTIFACT_DIR / 'oof_predictions.csv')

In [ ]:
# 本地验证：调用比赛评分逻辑计算整体 CV 分数

print('\n' + '=' * 60)
print('计算验证分数...')
print('=' * 60)

# 按照比赛评价方式计算本地交叉验证分数。
def calculate_cv_score(submission_df):
    """计算验证分数"""
    metadata_df = pl.read_csv(DATA_ROOT / 'train.csv').to_pandas()
    solution_segments = []
    for _, video_row in metadata_df.iterrows():
        lab_name = video_row['lab_id']
        if lab_name.startswith('MABe22'):
            continue
        video_key = video_row['video_id']
        file_path = TRAIN_LABEL_DIR / lab_name / f'{video_key}.parquet'
        try:
            annot = pd.read_parquet(file_path)
        except FileNotFoundError:
            continue
        annot['lab_id'] = lab_name
        annot['video_id'] = video_key
        annot['behaviors_labeled'] = video_row['behaviors_labeled']
        annot['target_id'] = np.where(annot.target_id != annot.agent_id, annot['target_id'].apply(lambda s: f'mouse{s}'), 'self')
        annot['agent_id'] = annot['agent_id'].apply(lambda s: f'mouse{s}')
        solution_segments.append(annot)
    solution_segments = pd.concat(solution_segments)
    submission_pd = submission_df.to_pandas() if isinstance(submission_df, pl.DataFrame) else submission_df
    scored_videos = set(submission_pd['video_id'].unique())
    solution_segments = solution_segments[solution_segments['video_id'].isin(scored_videos)]
    if len(solution_segments) == 0:
        return 0.0
    overall_f1 = score(solution_segments, submission_pd, 'row_id', beta=1.0)
    return overall_f1
oof_submission = pl.read_csv(ARTIFACT_DIR / 'oof_predictions.csv')
overall_f1 = calculate_cv_score(oof_submission)
print(f"\n{'=' * 60}")
print(f'Overall OOF F1 Score: {overall_f1:.4f}')
print(f"{'=' * 60}")
with open(ARTIFACT_DIR / 'overall_f1.txt', 'w') as f:
    f.write(f'{overall_f1}\n')
print('\n训练完成!')
print(f"模型保存在: {ARTIFACT_DIR / 'results'}")
print(f"OOF预测保存在: {ARTIFACT_DIR / 'oof_predictions.csv'}")

In [ ]:
# 指标分析：分别统计 self/pair 行为表现，辅助后续调参

# 输出 self 与 pair 行为的分组验证指标，便于定位问题。
def report_cv_metrics(submission_df, show_details=True):
    """Compute and display validation metrics for single vs pair behaviors."""
    metadata_df = pl.read_csv(DATA_ROOT / 'train.csv').to_pandas()
    solution_segments = []
    for _, video_row in metadata_df.iterrows():
        lab_name = video_row['lab_id']
        if lab_name.startswith('MABe22'):
            continue
        video_key = video_row['video_id']
        file_path = TRAIN_LABEL_DIR / lab_name / f'{video_key}.parquet'
        try:
            annot = pd.read_parquet(file_path)
        except FileNotFoundError:
            continue
        annot['lab_id'] = lab_name
        annot['video_id'] = video_key
        annot['behaviors_labeled'] = video_row['behaviors_labeled']
        annot['target_id'] = np.where(annot.target_id != annot.agent_id, annot['target_id'].apply(lambda s: f'mouse{s}'), 'self')
        annot['agent_id'] = annot['agent_id'].apply(lambda s: f'mouse{s}')
        solution_segments.append(annot)
    solution_segments = pd.concat(solution_segments)
    try:
        submission_single = submission_df[submission_df['target_id'] == 'self'].copy()
        submission_pair = submission_df[submission_df['target_id'] != 'self'].copy()
        scored_videos = set(submission_df['video_id'].unique())
        solution_segments = solution_segments[solution_segments['video_id'].isin(scored_videos)]
        if len(solution_segments) == 0:
            return
        overall_f1 = score(solution_segments, submission_df, 'row_id', beta=1.0)
        print(f"\n{'=' * 60}")
        print('PERFORMANCE METRICS')
        print(f"{'=' * 60}")
        print(f'Overall F1 Score: {overall_f1:.4f}')
        print(f'Total predictions: {len(submission_df)}')
        print(f'  - Single behaviors: {len(submission_single)}')
        print(f'  - Pair behaviors: {len(submission_pair)}')
        solution_polars = pl.DataFrame(solution_segments)
        submission_polars = pl.DataFrame(submission_df)
        solution_polars = solution_polars.with_columns(pl.concat_str([pl.col('video_id').cast(pl.Utf8), pl.col('agent_id').cast(pl.Utf8), pl.col('target_id').cast(pl.Utf8), pl.col('action')], separator='_').alias('label_key'))
        submission_polars = submission_polars.with_columns(pl.concat_str([pl.col('video_id').cast(pl.Utf8), pl.col('agent_id').cast(pl.Utf8), pl.col('target_id').cast(pl.Utf8), pl.col('action')], separator='_').alias('prediction_key'))
        action_stats = defaultdict(lambda: {'single': {'count': 0, 'f1': 0.0}, 'pair': {'count': 0, 'f1': 0.0}})
        for lab in solution_polars['lab_id'].unique():
            lab_solution = solution_polars.filter(pl.col('lab_id') == lab).clone()
            lab_videos = set(lab_solution['video_id'].unique())
            lab_submission = submission_polars.filter(pl.col('video_id').is_in(lab_videos)).clone()
            positive_frame_ids = defaultdict(set)
            prediction_frames = defaultdict(set)
            for video_row in lab_solution.to_dicts():
                positive_frame_ids[video_row['label_key']].update(range(video_row['start_frame'], video_row['stop_frame']))
            for video_row in lab_submission.to_dicts():
                key = video_row['prediction_key']
                prediction_frames[key].update(range(video_row['start_frame'], video_row['stop_frame']))
            for key in set(list(positive_frame_ids.keys()) + list(prediction_frames.keys())):
                action = key.split('_')[-1]
                behavior_mode = 'single' if 'self' in key else 'pair'
                pred_frames = prediction_frames.get(key, set())
                label_frames_set = positive_frame_ids.get(key, set())
                tp = len(pred_frames & label_frames_set)
                fn = len(label_frames_set - pred_frames)
                fp = len(pred_frames - label_frames_set)
                if tp + fn + fp > 0:
                    f1_value = (1 + 1 ** 2) * tp / ((1 + 1 ** 2) * tp + 1 ** 2 * fn + fp)
                    action_stats[action][behavior_mode]['count'] += 1
                    action_stats[action][behavior_mode]['f1'] += f1_value
        print('\nPer-Action Performance Summary:')
        print(f"{'-' * 60}")
        print(f"{'Action':<20} {'Mode':<10} {'Count':<10} {'Avg F1':<10}")
        print(f"{'-' * 60}")
        for action in sorted(action_stats.keys()):
            for behavior_mode in ['single', 'pair']:
                stats = action_stats[action][behavior_mode]
                if stats['count'] > 0:
                    avg_f1 = stats['f1'] / stats['count']
                    print(f"{action:<20} {behavior_mode:<10} {stats['count']:<10} {avg_f1:<10.4f}")
        single_actions = [a for a in action_stats.keys() if action_stats[a]['single']['count'] > 0]
        pair_actions = [a for a in action_stats.keys() if action_stats[a]['pair']['count'] > 0]
        if single_actions:
            single_avg_f1 = np.mean([action_stats[a]['single']['f1'] / action_stats[a]['single']['count'] for a in single_actions if action_stats[a]['single']['count'] > 0])
            print(f'\nSingle behaviors: {len(single_actions)} actions, Avg F1: {single_avg_f1:.4f}')
        if pair_actions:
            pair_avg_f1 = np.mean([action_stats[a]['pair']['f1'] / action_stats[a]['pair']['count'] for a in pair_actions if action_stats[a]['pair']['count'] > 0])
            print(f'Pair behaviors: {len(pair_actions)} actions, Avg F1: {pair_avg_f1:.4f}')
        print(f"{'=' * 60}\n")
    except Exception as e:
        if show_details:
            error_msg = str(e)
            if len(error_msg) > 200:
                error_msg = error_msg[:200] + '...'
            print(f'\nWarning: Could not compute validation metrics: {error_msg}')
            if show_details:
                print(f'Traceback: {traceback.format_exc()[:300]}')
report_cv_metrics(submission_df=pd.read_csv(ARTIFACT_DIR / 'oof_predictions.csv'))